In [ ]:
# ============================================================
# SignLingo Round 2 -- Phase 3: train_final
# ============================================================
# Trains the ONE deployable model on the arm that won the ablation, then produces
# all three paper tables from that single set of weights in a single process:
#
#   Table I    architecture and parameter footprint
#   Table II   test-set metrics
#   Table III  quantization comparison, four TFLite formats
#
# Doing this in one process is the point. The thesis previously chased three
# different accuracy figures (99.32 / 99.55 / 99.77) because each script loaded a
# different model at a different time and tuner_results.py retrained from scratch
# on every run. Here the weights that produce Table I also produce Tables II and
# III, so the three cannot disagree.
#
# WHICH NUMBER MEANS WHAT. Table II is measured on a sequence-level split, so it
# answers "can it recognise a new recording from a signer it has already seen?"
# It is NOT the signer-independent number. That one is the LOSO mean from Phase 2,
# and both belong in the paper, clearly labelled. Reporting Table II alone is the
# exact overclaim this whole round exists to correct.
import os
import json
import shutil
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Input
from tensorflow.keras import regularizers
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, log_loss, confusion_matrix)

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "font.family": "DejaVu Sans"})

In [ ]:
# ============================================================
# 0 -- Config
# ============================================================
DATA_PATH     = "dataset.npz"
ABLATION_PATH = "phase2_results.json"
MODEL_PATH    = "models/signlingo_r2_best.h5"
SAVED_MODEL   = "signlingo_r2_saved_model_tmp"
RESULTS_PATH  = "final_results.json"
RANDOM_STATE  = 42

# Identical to run_ablation. If these drift, Table II stops describing the model
# the ablation actually selected.
GRU1, GRU2, DROPOUT, L2_RATE, USE_REC_L2, LR = 32, 256, 0.3, 1e-4, False, 1e-3
EPOCHS       = 150
BATCH_SIZE   = 32
PATIENCE     = 15
AUG_FACTOR   = 3
MIRROR_PROB  = 0.5
CALIB_SIZE   = 200          # representative samples for full-INT8 calibration

# Set to override the ablation winner, e.g. to deploy a smaller arm that scored
# within noise of the best. None means "use whatever Phase 2 selected".
ARM_OVERRIDE = None

TOY_MODE = False
if TOY_MODE:
    GRU1, GRU2, EPOCHS, PATIENCE, AUG_FACTOR = 4, 8, 1, 1, 2
    CALIB_SIZE = 20
    MODEL_PATH, RESULTS_PATH = "models/signlingo_r2_toy.h5", "final_results_toy.json"
    # run_ablation in toy mode writes the _toy results file, so follow it here
    # or the integration pass reads a real ablation that does not exist yet.
    ABLATION_PATH = "phase2_results_toy.json"

# Quantization can take the whole process down: a Flex-delegate TFLite interpreter
# segfaulted here once, on a code path that had already succeeded on the previous
# run. Training is the expensive stage and runs first, so every stage checkpoints
# to RESULTS_PATH and a re-run resumes instead of retraining for an hour.
results = {}
if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH) as fh:
        results = json.load(fh)
    done = [k for k in ("training", "table_i", "table_ii", "table_iii")
            if results.get(k)]
    print("Resuming from %s (done: %s)" % (RESULTS_PATH, ", ".join(done) or "nothing"))


def save_results():
    with open(RESULTS_PATH, "w") as fh:
        json.dump(results, fh, indent=2)

In [ ]:
# ============================================================
# 1 -- Load data and the ablation verdict
# ============================================================
d = np.load(DATA_PATH, allow_pickle=True)
X           = d["X"].astype(np.float32)
y_int       = d["y"].astype(int)
class_names = [str(c) for c in d["class_names"]]
nms_mask    = d["nms_mask"]
mirror_perm = d["mirror_perm"]
mirror_sign = d["mirror_sign"].astype(np.float32)
num_classes = len(class_names)
y_oh = to_categorical(y_int, num_classes).astype(np.float32)

# The winning arm is read from disk, not typed in here, so Phase 2 and Phase 3
# chain without a human editing a constant between them.
abl = None
if os.path.exists(ABLATION_PATH):
    with open(ABLATION_PATH) as fh:
        abl = json.load(fh)

if ARM_OVERRIDE is not None:
    arm_name = ARM_OVERRIDE
    arm_cut = dict(zip([str(a) for a in d["arm_names"]],
                       [int(c) for c in d["arm_cuts"]]))[arm_name]
    print("Arm OVERRIDDEN by hand: %s (%d dims)" % (arm_name, arm_cut))
elif abl and abl.get("best_arm"):
    arm_name = abl["best_arm"]["name"]
    arm_cut = int(abl["best_arm"]["cut"])
    print("Arm selected by the ablation: %s (%d dims), %.2f%% mean LOSO"
          % (arm_name, arm_cut, abl["best_arm"]["mean_accuracy"] * 100))
else:
    raise FileNotFoundError(
        "%s has no best_arm. Run run_ablation first, or set ARM_OVERRIDE to pick "
        "an arm by hand." % ABLATION_PATH)

# Guard against training a model the ablation never actually measured.
if abl:
    cfg = abl.get("config", {})
    drift = {k: (cfg.get(k), v) for k, v in
             (("gru1", GRU1), ("gru2", GRU2), ("dropout", DROPOUT),
              ("l2", L2_RATE), ("lr", LR)) if k in cfg and cfg[k] != v}
    assert not drift, (
        "hyperparameters differ from the ablation run (%s). Table II would then "
        "describe a model the ablation never evaluated." % drift)
    if cfg.get("toy_mode") and not TOY_MODE:
        print("  WARNING: %s came from a TOY run. Its arm choice is meaningless."
              % ABLATION_PATH)

# A resumed run must not silently switch arms underneath its own saved tables.
if results.get("arm") and results["arm"] != arm_name:
    raise AssertionError(
        "%s holds results for arm %s but this run selected %s. Delete the file to "
        "start clean." % (RESULTS_PATH, results["arm"], arm_name))

Xa = X[..., :arm_cut]
print("Training on %s: %d sequences, %d classes, %d dims"
      % (arm_name, len(Xa), num_classes, arm_cut))

In [ ]:
# ============================================================
# 2 -- Split, augment, train
# ============================================================
# Sequence-level 80/10/10, stratified by class. This is deliberately the protocol
# the original paper used, so Table II is comparable to the published 99.32%.
# It is a seen-signer measurement; the unseen-signer number is Phase 2's LOSO.
tr, tmp = train_test_split(np.arange(len(Xa)), test_size=0.2,
                           stratify=y_int, random_state=RANDOM_STATE)
va, te = train_test_split(tmp, test_size=0.5,
                          stratify=y_int[tmp], random_state=RANDOM_STATE)
print("Split: %d train / %d val / %d test" % (len(tr), len(va), len(te)))

A_END, B_END = 126, 138
I_TILT, I_ROLL = 138, 139


def mirror(a):
    """Signed permutation from prepare_features, sliced to this arm's width."""
    p, s = mirror_perm[:arm_cut], mirror_sign[:arm_cut]
    assert p.max() < arm_cut, "this arm splits a mirror pair; cuts must be block-aligned"
    return a[..., p] * s


def augment_sequence(seq, rng):
    """Block-aware, identical to run_ablation. Only block A is coordinates; the
    rest are unit vectors, angles and scale-free ratios, so a uniform scale or a
    naive 3D rotation over the whole vector would be wrong. The width guards let
    this run unchanged on a narrow arm, e.g. hands-only at 126 dims."""
    n_frames, n_feat = seq.shape
    out = seq.astype(np.float32).copy()

    new_len = n_frames * rng.uniform(0.9, 1.1)
    src = np.linspace(0, n_frames - 1, n_frames)
    dst = np.clip(np.linspace(0, new_len - 1, n_frames), 0, n_frames - 1)
    out = np.stack([np.interp(dst, src, out[:, f]) for f in range(n_feat)], axis=1)

    th = np.radians(rng.uniform(-5, 5))
    c, s = np.cos(th), np.sin(th)
    R2 = np.array([[c, -s], [s, c]], dtype=np.float32)
    for lo, hi in ((0, min(A_END, n_feat)), (A_END, min(B_END, n_feat))):
        if hi <= lo:
            continue
        v = out[:, lo:hi].reshape(n_frames, -1, 3)
        v[..., :2] = v[..., :2] @ R2.T
        out[:, lo:hi] = v.reshape(n_frames, hi - lo)
    if n_feat > I_TILT:
        out[:, I_TILT] += th
    if n_feat > I_ROLL:
        out[:, I_ROLL] += th

    out[:, 0:min(A_END, n_feat)] *= np.float32(rng.uniform(0.95, 1.05))
    out += rng.normal(0, 0.005, out.shape).astype(np.float32)
    if n_feat >= B_END:
        v = out[:, A_END:B_END].reshape(n_frames, 4, 3)
        n = np.linalg.norm(v, axis=-1, keepdims=True)
        out[:, A_END:B_END] = (v / np.maximum(n, 1e-6)).reshape(n_frames, B_END - A_END)
    return out.astype(np.float32)


def build_model(unroll=False):
    """unroll only changes how the 30-step time loop is emitted, never a
    parameter, so weights transfer between the two forms exactly (verified at
    max|diff| = 0.0). Training uses the rolled form; export uses the unrolled
    one, for the reason explained at the TFLite section."""
    return Sequential([
        Input(shape=(Xa.shape[1], arm_cut)),
        GRU(GRU1, return_sequences=True, name="gru_1", unroll=unroll,
            kernel_regularizer=regularizers.l2(L2_RATE),
            recurrent_regularizer=regularizers.l2(L2_RATE) if USE_REC_L2 else None),
        Dropout(DROPOUT, name="dropout_1"),
        GRU(GRU2, return_sequences=False, name="gru_2", unroll=unroll,
            kernel_regularizer=regularizers.l2(L2_RATE),
            recurrent_regularizer=regularizers.l2(L2_RATE) if USE_REC_L2 else None),
        Dropout(DROPOUT, name="dropout_2"),
        Dense(num_classes, activation="softmax", name="output",
              kernel_regularizer=regularizers.l2(L2_RATE)),
    ], name="SignLingo_GRU_R2_final")


if os.path.exists(MODEL_PATH) and results.get("training"):
    model = tf.keras.models.load_model(MODEL_PATH)
    history = results["training"]["history"]
    train_minutes = results["training"]["minutes"]
    best_epoch = results["training"]["best_epoch"]
    print("Reusing the model already trained at %s. Delete it to force a retrain."
          % MODEL_PATH)
else:
    rng = np.random.default_rng(RANDOM_STATE)
    pool_x, pool_y = [Xa[tr]], [y_oh[tr]]
    for _ in range(max(0, AUG_FACTOR - 1)):
        aug = np.stack([augment_sequence(s, rng) for s in Xa[tr]])
        flip = rng.random(len(aug)) < MIRROR_PROB
        aug[flip] = mirror(aug[flip])
        pool_x.append(aug)
        pool_y.append(y_oh[tr])
    Xtr, ytr = np.concatenate(pool_x), np.concatenate(pool_y)
    idx = rng.permutation(len(Xtr))
    Xtr, ytr = Xtr[idx], ytr[idx]
    print("Training pool after x%d augmentation: %d sequences" % (AUG_FACTOR, len(Xtr)))

    tf.keras.backend.clear_session()
    model = build_model()
    model.compile(optimizer=Adam(learning_rate=LR),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    hist = model.fit(Xtr, ytr, validation_data=(Xa[va], y_oh[va]),
                     epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2,
                     callbacks=[
                         EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True),
                         ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                           patience=6, min_lr=1e-6, verbose=0),
                     ])
    train_minutes = round((time.time() - t0) / 60, 2)
    history = {k: [float(x) for x in v] for k, v in hist.history.items()}
    best_epoch = int(np.argmin(history["val_loss"])) + 1
    os.makedirs(os.path.dirname(MODEL_PATH) or ".", exist_ok=True)
    model.save(MODEL_PATH)

    results.update({
        "arm": arm_name, "dims": arm_cut, "model_path": MODEL_PATH,
        "selected_by": "ARM_OVERRIDE" if ARM_OVERRIDE else "ablation best_arm",
        "config": {"gru1": GRU1, "gru2": GRU2, "dropout": DROPOUT, "l2": L2_RATE,
                   "use_recurrent_l2": USE_REC_L2, "lr": LR,
                   "aug_factor": AUG_FACTOR, "toy_mode": TOY_MODE},
        "training": {"epochs_run": len(history["loss"]), "best_epoch": best_epoch,
                     "best_val_loss": min(history["val_loss"]),
                     "best_val_accuracy": max(history["val_accuracy"]),
                     "minutes": train_minutes, "history": history},
    })
    save_results()

print("\nStopped at epoch %d, best weights from epoch %d (val_loss %.4f), %.1f min"
      % (len(history["loss"]), best_epoch, min(history["val_loss"]), train_minutes))
print("Model at %s" % MODEL_PATH)

In [ ]:
# ============================================================
# 3 -- TABLE I: architecture and parameter footprint
# ============================================================
model.summary()
results["table_i"] = [{"layer": l.name, "output_shape": str(l.output.shape),
                       "params": int(l.count_params())} for l in model.layers]
results["total_params"] = int(model.count_params())
save_results()

print("\nTABLE I  Architecture")
print("%-14s %-22s %12s" % ("layer", "output shape", "params"))
for r in results["table_i"]:
    print("%-14s %-22s %12s" % (r["layer"], r["output_shape"], format(r["params"], ",")))
print("%-14s %-22s %12s" % ("TOTAL trainable", "", format(model.count_params(), ",")))
# summary() prints a larger "Total params" because it also counts optimizer slot
# variables. The paper reports trainable parameters, the figure above, which is
# what the old paper's 279,176 referred to.

In [ ]:
# ============================================================
# 4 -- TABLE II: test-set metrics
# ============================================================
# Cross-entropy is computed with sklearn on the probabilities, NOT taken from
# model.evaluate(). evaluate() adds the L2 penalty term to the reported loss, so
# it prints a larger number that is not the cross-entropy the paper reports.
prob = model.predict(Xa[te], verbose=0)
pred, true = prob.argmax(1), y_int[te]
results["table_ii"] = {
    "arm": arm_name, "dims": arm_cut, "n_test": int(len(te)),
    "accuracy":  float(accuracy_score(true, pred)),
    "precision": float(precision_score(true, pred, average="weighted", zero_division=0)),
    "recall":    float(recall_score(true, pred, average="weighted", zero_division=0)),
    "f1":        float(f1_score(true, pred, average="weighted", zero_division=0)),
    "cross_entropy": float(log_loss(true, prob, labels=list(range(num_classes)))),
    "keras_evaluate_loss": float(model.evaluate(Xa[te], y_oh[te], verbose=0)[0]),
    "acc_nms": float(accuracy_score(true[nms_mask[true]], pred[nms_mask[true]])),
    "acc_manual": float(accuracy_score(true[~nms_mask[true]], pred[~nms_mask[true]])),
    "pred": pred.tolist(), "true": true.tolist(),
}
t2 = results["table_ii"]
results["loso_mean_for_this_arm"] = (abl["arms"][arm_name]["mean_accuracy"]
                                     if abl and abl.get("arms", {}).get(arm_name)
                                     else None)
save_results()

print("\nTABLE II  Test set (%d sequences, sequence-level split)" % len(te))
for k in ("accuracy", "precision", "recall", "f1"):
    print("  %-16s %.4f" % (k, t2[k]))
print("  %-16s %.4f   <- report this one" % ("cross_entropy", t2["cross_entropy"]))
print("  %-16s %.4f   (includes the L2 penalty, do NOT report)"
      % ("evaluate() loss", t2["keras_evaluate_loss"]))
print("  NMS-focused %.2f%%   standard-manual %.2f%%"
      % (t2["acc_nms"] * 100, t2["acc_manual"] * 100))
if results["loso_mean_for_this_arm"] is not None:
    print("\n  Seen-signer (above):    %.2f%%" % (t2["accuracy"] * 100))
    print("  Unseen-signer (LOSO):   %.2f%%  <- the generalization claim"
          % (results["loso_mean_for_this_arm"] * 100))
    print("  Both belong in the paper. The first alone is the overclaim Round 2 fixes.")

In [ ]:
# ============================================================
# 5 -- TABLE III: quantization
# ============================================================
# Conversion logic carried over from quantization.py unchanged, including the
# Flex-delegate workaround: the converter's tensor-list lowering pass needs a
# static element_shape that Keras 3's SavedModel export does not provide for GRU,
# so SELECT_TF_OPS handles that one op. Consequence for the paper: the quantized
# models are "mostly INT8" and need the Flex delegate at runtime, so pure
# microcontroller INT8 deployment is not achievable for GRU with current TFLite.
#
# Each format is checkpointed as it completes, so the segfault that motivated the
# resume logic costs one format rather than the whole run.
if os.path.exists(SAVED_MODEL):
    shutil.rmtree(SAVED_MODEL)

# Exported UNROLLED. A rolled GRU emits TensorList ops that TFLite cannot lower,
# so it converts only with SELECT_TF_OPS and then needs the Flex delegate at run
# time. The Flex delegate is compiled into the TensorFlow build rather than
# installed separately, and macOS ARM wheels ship without it, so a rolled model
# converts on any machine but fails at allocate_tensors() on some of them.
#
# Unrolling the 30 steps removes the TensorList ops entirely, giving pure TFLite
# builtins that run anywhere. It costs file size, measured on this architecture:
#
#     format          rolled + Flex     unrolled
#     Float32              990 KB       1165 KB
#     Float16              506 KB        680 KB
#     Dynamic Range        283 KB        458 KB
#     Full INT8            289 KB        568 KB
#
# That trade is worth taking for deployment. The rolled file is smaller but its
# runtime is not: the Flex delegate pulls in a large slice of TensorFlow, so the
# real on-device footprint is far bigger than the model file. The unrolled model
# needs only the TFLite interpreter, and it makes true INT8-only deployment
# achievable, which the Round 1 write-up had to list as a limitation.
export_model = build_model(unroll=True)
export_model.set_weights(model.get_weights())
_chk = np.abs(export_model.predict(Xa[te][:16], verbose=0)
              - model.predict(Xa[te][:16], verbose=0)).max()
assert _chk < 1e-5, "unrolled export does not match the trained model (%.2e)" % _chk
print("Unrolled export matches the trained model (max|diff| %.1e)" % _chk)
export_model.export(SAVED_MODEL)


def _ops(conv, base_ops, allow_flex=False):
    """Pure builtins by default. allow_flex is the fallback path only."""
    conv.target_spec.supported_ops = (base_ops + [tf.lite.OpsSet.SELECT_TF_OPS]
                                      if allow_flex else base_ops)
    if allow_flex:
        conv._experimental_lower_tensor_list_ops = False
    return conv


calib = Xa[te][:min(CALIB_SIZE, len(te))].astype(np.float32)


def _rep():
    for s in calib:
        yield [s[np.newaxis, ...]]


def _fp32(c, flex=False):
    """Unquantized baseline: every later row's size reduction is measured off this."""
    _ops(c, [tf.lite.OpsSet.TFLITE_BUILTINS], flex)


def _fp16(c, flex=False):
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.target_spec.supported_types = [tf.float16]
    _ops(c, [tf.lite.OpsSet.TFLITE_BUILTINS], flex)


def _dynamic(c, flex=False):
    """INT8 weights, FP32 activations. No representative dataset, by definition."""
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    _ops(c, [tf.lite.OpsSet.TFLITE_BUILTINS], flex)


def _int8(c, flex=False):
    """INT8 weights and activations. Needs calibration data to fix activation
    ranges. I/O stays FP32 so callers need not quantize their own input."""
    c.optimizations = [tf.lite.Optimize.DEFAULT]
    c.representative_dataset = _rep
    _ops(c, [tf.lite.OpsSet.TFLITE_BUILTINS_INT8], flex)
    c.inference_input_type = tf.float32
    c.inference_output_type = tf.float32


FORMATS = [
    ("Float32 (baseline)", "signlingo_r2_fp32.tflite", _fp32),
    ("Float16", "signlingo_r2_fp16.tflite", _fp16),
    ("Dynamic Range", "signlingo_r2_dynamic.tflite", _dynamic),
    ("Full INT8", "signlingo_r2_int8.tflite", _int8),
]


def tflite_accuracy(path, Xe, y_true):
    it = tf.lite.Interpreter(model_path=path)
    it.allocate_tensors()
    i_d, o_d = it.get_input_details()[0], it.get_output_details()[0]
    preds = []
    for s in Xe:
        it.set_tensor(i_d["index"], s[np.newaxis, ...].astype(np.float32))
        it.invoke()
        preds.append(int(np.argmax(it.get_tensor(o_d["index"]))))
    return float(accuracy_score(y_true, preds))


def kb(path):
    return round(os.path.getsize(path) / 1024.0, 1)


rows = {r["format"]: r for r in results.get("table_iii", [])}
rows["Raw Keras .h5 (eager)"] = {"format": "Raw Keras .h5 (eager)",
                                 "accuracy": t2["accuracy"],
                                 "size_kb": kb(MODEL_PATH), "reduction_pct": None}
for label, path, configure in FORMATS:
    if label in rows and os.path.exists(path):
        print("  %s already converted, skipping." % label)
        continue
    print("Converting: %s ..." % label)
    # Pure builtins first. The fallback exists so an unexpected op cannot block
    # the run outright, but it is loud, because a Flex-dependent file will fail
    # at allocate_tensors() on any machine whose TensorFlow build lacks the
    # delegate, which includes typical macOS ARM wheels.
    used_flex = False
    try:
        conv = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
        configure(conv, False)
        blob = conv.convert()
    except Exception as exc:
        print("  pure-builtin conversion failed (%s); retrying with the Flex"
              % type(exc).__name__)
        print("  delegate. THIS FILE WILL NOT RUN WHERE THE DELEGATE IS ABSENT.")
        conv = tf.lite.TFLiteConverter.from_saved_model(SAVED_MODEL)
        configure(conv, True)
        blob = conv.convert()
        used_flex = True
    with open(path, "wb") as fh:
        fh.write(blob)
    rows[label] = {"format": label, "path": path, "size_kb": kb(path),
                   "needs_flex_delegate": used_flex,
                   "accuracy": tflite_accuracy(path, Xa[te], true)}
    results["table_iii"] = [rows[k] for k in
                            ["Raw Keras .h5 (eager)"] + [f[0] for f in FORMATS]
                            if k in rows]
    save_results()

table_iii = results["table_iii"]
fp32_kb = next(r["size_kb"] for r in table_iii if r["format"].startswith("Float32"))
for r in table_iii[1:]:
    r["reduction_pct"] = round((1 - r["size_kb"] / fp32_kb) * 100, 1)
save_results()

print("\nTABLE III  Quantization (%d test sequences)" % len(te))
print("%-24s %10s %11s %12s" % ("format", "accuracy", "size (KB)", "vs FP32"))
for r in table_iii:
    print("%-24s %9.2f%% %11.1f %12s"
          % (r["format"], r["accuracy"] * 100, r["size_kb"],
             "-" if r["reduction_pct"] is None else "%.1f%%" % r["reduction_pct"]))

lossless = [r for r in table_iii[1:] if r["accuracy"] >= t2["accuracy"] - 1e-9]
rec = min(lossless or table_iii[1:], key=lambda r: r["size_kb"])
results["recommended_format"] = rec["format"]
save_results()
print("\nRecommended: %s, %.1f KB (%.1f%% smaller than FP32), accuracy %.2f%%"
      % (rec["format"], rec["size_kb"], rec["reduction_pct"], rec["accuracy"] * 100))
print("Note: model file size only. It excludes the TFLite runtime and the Flex")
print("delegate that the GRU op requires, so it is not the on-device footprint.")
print("\nInference latency is NOT measured here. Run latency.ipynb, which times the")
print("real capture loop including the compact derivation, on a machine with a webcam.")

In [ ]:
# ============================================================
# 6 -- Charts
# ============================================================
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].plot(history["accuracy"], label="train")
ax[0].plot(history["val_accuracy"], label="val")
ax[0].axvline(best_epoch - 1, color="#C44E52", ls="--", lw=1, label="best")
ax[0].set_xlabel("epoch")
ax[0].set_ylabel("accuracy")
ax[0].legend(fontsize=8)
ax[0].set_title("Training curve", fontweight="bold")

ax[1].plot(history["loss"], label="train")
ax[1].plot(history["val_loss"], label="val")
ax[1].axvline(best_epoch - 1, color="#C44E52", ls="--", lw=1)
ax[1].set_xlabel("epoch")
ax[1].set_ylabel("loss")
ax[1].legend(fontsize=8)
ax[1].set_title("Loss", fontweight="bold")

q = table_iii[1:]
ax[2].bar([r["format"].split()[0] for r in q], [r["size_kb"] for r in q],
          color="#4C72B0")
for i, r in enumerate(q):
    ax[2].text(i, r["size_kb"], "%.1f%%" % (r["accuracy"] * 100),
               ha="center", va="bottom", fontsize=8)
ax[2].set_ylabel("size (KB)")
ax[2].set_title("Quantization: size, labelled with accuracy", fontweight="bold")
ax[2].tick_params(axis="x", rotation=15, labelsize=8)
plt.tight_layout()
plt.savefig("final_training.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(confusion_matrix(true, pred, labels=list(range(num_classes))),
            ax=ax, cmap="mako", square=True,
            xticklabels=class_names, yticklabels=class_names)
ax.set_title("Confusion matrix, %s, held-out test set" % arm_name, fontweight="bold")
ax.set_xlabel("predicted")
ax.set_ylabel("true")
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6, rotation=0)
plt.tight_layout()
plt.savefig("final_confusion.png", dpi=120)
plt.show()

print("\nWrote %s, %s, final_training.png, final_confusion.png"
      % (RESULTS_PATH, MODEL_PATH))
print("TFLite: " + ", ".join(r["path"] for r in table_iii[1:]))